# TSFM-PDM Industrial Benchmarking — Unified Colab Notebook
**Author:** Yassire Ammouri  
**Paper:** *Benchmarking Time-Series Foundation Models for Industrial Predictive Maintenance* (ICATH 2025)

---

| Step | Content |
|------|---------|
| 1 | Environment setup — GPU check, clone repo, install packages |
| 2 | Dataset download — C-MAPSS (NASA/Kaggle), Wind SCADA (Kaggle), PHM Milling (Kaggle/manual) |
| 3 | Data preprocessing — chronological splits, per-sensor normalization |
| 4 | **Zero-shot experiments** — MOMENT · Chronos · Lag-Llama · PatchTST × 3 datasets |
| 5 | **Few-shot LoRA** — MOMENT + Lag-Llama × 3 datasets (1 % training data, r=16) |
| 6 | **Cross-condition transfer** — C-MAPSS FD001 → FD001 / FD002 / FD003 / FD004 |
| 7 | Results aggregation — DataFrames + pivot tables |
| 8 | Inference benchmarking + CSV export of all results |
| 9 | Figures — bar charts, ZS vs FS comparison, cross-condition heatmap |
| 10 | Export everything to Google Drive |

**Estimated runtime on Colab Free T4:** 8–12 hours  
> **Before running:** Runtime → Change runtime type → **T4 GPU**

## ⚙️ Section 0 — Configuration
Edit the variables below before executing the notebook.

In [ ]:
# ================================================================
# USER CONFIGURATION — edit before running
# ================================================================
from google.colab import userdata
# GitHub repository URL (set after you push the repo)
GITHUB_REPO = "https://github.com/Yassire1/icath-article.git"

# Google Drive folder to save checkpoints + results (leave empty to skip)
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/tsfm_pdm_bench/results"

# Kaggle API credentials (needed for Wind SCADA + optional C-MAPSS / PHM download)
# Alternatively upload kaggle.json to /root/.kaggle/ via Files panel
KAGGLE_USERNAME = "ammouriyassire"
KAGGLE_KEY = userdata.get('kaggle_api')

# Experiments to run
RUN_ZERO_SHOT       = True
RUN_FEW_SHOT        = True
RUN_CROSS_CONDITION = True

# Models
MODELS_ZERO_SHOT = ["moment", "chronos", "lag_llama", "patchtst"]
MODELS_FEW_SHOT  = ["moment", "lag_llama"]   # only LoRA-capable models

# Datasets
DATASETS = ["cmapss", "wind_scada", "phm_milling"]

# C-MAPSS subsets (FD001 always required; add FD002-FD004 for cross-condition)
CMAPSS_SUBSETS = ["FD001", "FD002", "FD003", "FD004"]

# Preprocessing
CMAPSS_LOOKBACK       = 64    # shorter context fits short engine sequences
CMAPSS_HORIZON        = 30
LOOKBACK              = 512   # Wind SCADA
HORIZON               = 96
PHM_MILLING_LOOKBACK  = 128   # cut-level summary sequence
PHM_MILLING_HORIZON   = 16

# Few-shot LoRA
LORA_R           = 16
LORA_ALPHA       = 32
LORA_EPOCHS      = 2
LORA_LR          = 1e-4
TRAIN_RATIO      = 0.01  # 1 % of training data

# Misc
SEED       = 42
DEVICE     = "cuda"
USE_WANDB  = False
WANDB_PROJECT = "tsfm-pdm-bench"


## 🖥️ Section 1 — Environment Setup

In [ ]:
# 1-A  GPU check
import subprocess, sys, os

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout[:600] if result.returncode == 0 else "nvidia-smi not available")

import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    DEVICE = "cpu"
    print("WARNING: No GPU found — CPU inference will be very slow.")


In [ ]:
# 1-B  Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
    print(f"Drive mounted. Results will be copied to: {DRIVE_RESULTS_DIR}")
except Exception:
    DRIVE_AVAILABLE = False
    print("Not in Colab / Drive not mounted. Results saved locally only.")


In [ ]:
# 1-C  Clone repository
REPO_DIR = "/content/tsfm-pdm-bench"

if not os.path.exists(REPO_DIR):
    if "YOUR_USERNAME" in GITHUB_REPO:
        raise ValueError(
            "Please set GITHUB_REPO in the config cell above.\n"
            "Example: GITHUB_REPO = 'https://github.com/yourusername/tsfm-pdm-bench.git'"
        )
    print(f"Cloning {GITHUB_REPO} …")
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
else:
    print(f"Repo already at {REPO_DIR} — pulling latest …")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Working directory: {os.getcwd()}")


In [ ]:
# 1-D  Install Python dependencies  (5-10 min on first run)
import subprocess, sys

def pip_install(args):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + args, check=False)

# Keep the scientific stack on one NumPy ABI to avoid
# "numpy.dtype size changed" binary incompatibility errors.
core_stack = [
    '--upgrade', '--force-reinstall',
    'numpy<2.0', 'pandas', 'scipy', 'scikit-learn',
]
pip_install(core_stack)

pkgs = [
    "momentfm",
    "chronos-forecasting>=1.3.0",
    "gluonts>=0.15.0", "pytorch-lightning>=2.0.0",
    "peft>=0.12.0", "transformers>=4.44.0", "accelerate>=0.33.0", "huggingface_hub",
    "neuralforecast>=1.7.0",
    "kaggle",
    "wandb", "torchmetrics",
    "numpy<2.0",
]
pip_install(['--upgrade'] + pkgs)

# Lag-Llama from GitHub
pip_install(
    ['--upgrade', 'git+https://github.com/time-series-foundation-models/lag-llama.git']
)

# Reinstall core stack at the end so compiled extensions
# definitely match the active NumPy version.
pip_install(core_stack)

print("Package installation finished.")
print()
print("*** IMPORTANT: Runtime -> Restart session, then re-run all cells from the top. ***")
print("    (compiled wheels were reinstalled; interpreter restart is required)")


In [ ]:
# 1-E  Verify imports + set seeds
import importlib.util, random, subprocess, sys
import numpy as np

try:
    import pandas as pd
    import scipy
    import sklearn
except Exception as e:
    raise RuntimeError(
        "Binary dependency import failed. Re-run the install dependencies cell, restart runtime, and run all cells from the top.\n"
        f"Original error: {e}"
    )

print(
    f"Versions -> numpy={np.__version__}, pandas={pd.__version__}, scipy={scipy.__version__}, sklearn={sklearn.__version__}"
)

critical_modules = {
    "momentfm": "momentfm",
    "chronos": "chronos-forecasting",
    "neuralforecast": "neuralforecast",
}

def _find_missing(module_names):
    return [m for m in module_names if importlib.util.find_spec(m) is None]

missing_critical = _find_missing(critical_modules.keys())
if missing_critical:
    print("Missing critical packages detected. Attempting repair...")
    for mod in missing_critical:
        pkg = critical_modules[mod]
        print(f"  Installing {pkg} ...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', pkg], check=False)

    # Re-check and try a no-deps fallback to avoid forcing NumPy upgrades.
    still_missing = _find_missing(critical_modules.keys())
    if still_missing:
        print("Retrying with --no-deps for remaining packages...")
        for mod in still_missing:
            pkg = critical_modules[mod]
            print(f"  Installing {pkg} (--no-deps) ...")
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', pkg], check=False)

    still_missing = _find_missing(critical_modules.keys())
    if still_missing:
        missing_pkgs = [critical_modules[m] for m in still_missing]
        raise RuntimeError(
            "Could not import required packages after repair: "
            + ", ".join(missing_pkgs)
            + ". Re-run Cell 8, restart runtime, then run Cell 9 again."
        )
    print("Critical package repair completed.")

packages_check = {
    "torch":           "PyTorch",
    "momentfm":        "MOMENT (momentfm)",
    "chronos":         "Chronos",
    "lag_llama":       "Lag-Llama",
    "neuralforecast":  "NeuralForecast (PatchTST)",
    "peft":            "PEFT / LoRA",
    "gluonts":         "GluonTS",
    "wandb":           "WandB",
}
print("Package status:")
for pkg, label in packages_check.items():
    ok = importlib.util.find_spec(pkg) is not None
    print(f"  {'✓' if ok else '✗'} {label}")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"\nSeed: {SEED}  |  Device: {DEVICE}")


## 📦 Section 2 — Data Download

### Dataset sources
| Dataset | Size | Download method |
|---------|------|-----------------|
| **C-MAPSS** (NASA Turbofan) | ~2 MB | NASA portal **or** Kaggle (auto) |
| **Wind SCADA** | ~5 MB | Kaggle API (auto, needs credentials) |
| **PHM Milling** (CNC tool wear) | ~800 MB per cutter archive | Kaggle mirror (auto) or PHM Society (manual) |

### C-MAPSS — two options
**Option A (recommended):** set `KAGGLE_USERNAME` / `KAGGLE_KEY` above → auto-downloaded below.  
**Option B (manual):** download `CMAPSSData.zip` from  
https://ti.arc.nasa.gov/tech/dash/groups/pcoe/prognostic-data-repository/#turbofan  
then upload/copy to `/content/tsfm-pdm-bench/data/raw/cmapss/`.


In [ ]:
# 2-A  Create all raw data directories
from pathlib import Path

RAW_DIR  = Path("data/raw")
PROC_DIR = Path("data/processed")
RES_DIR  = Path("results")

for d in [
    RAW_DIR / "cmapss",
    RAW_DIR / "wind_scada",
    RAW_DIR / "phm_milling",
    PROC_DIR / "cmapss",
    PROC_DIR / "wind_scada",
    PROC_DIR / "phm_milling",
    RES_DIR / "zero_shot",
    RES_DIR / "few_shot",
    RES_DIR / "cross_condition",
    RES_DIR / "tables",
    RES_DIR / "figures",
]:
    d.mkdir(parents=True, exist_ok=True)

print("Directory tree created.")


In [ ]:
# 2-B  Configure Kaggle API
import os, json
from pathlib import Path

KAGGLE_DIR = Path("/root/.kaggle")
KAGGLE_DIR.mkdir(exist_ok=True)
kaggle_json = KAGGLE_DIR / "kaggle.json"

if not kaggle_json.exists():
    if KAGGLE_USERNAME and KAGGLE_KEY:
        kaggle_json.write_text(
            json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY})
        )
        kaggle_json.chmod(0o600)
        print("Kaggle credentials written from config variables.")
    else:
        print("WARNING: Kaggle credentials not set.")
        print("  Either set KAGGLE_USERNAME / KAGGLE_KEY in the config cell,")
        print("  or upload kaggle.json via Files panel → /root/.kaggle/kaggle.json")
else:
    print("Kaggle credentials found at /root/.kaggle/kaggle.json")


In [ ]:
# 2-C  Download C-MAPSS (Kaggle, dataset: behrad3d/nasa-cmaps)
import subprocess
from pathlib import Path

CMAPSS_RAW = Path("data/raw/cmapss")
sentinel = CMAPSS_RAW / "train_FD001.txt"

if sentinel.exists():
    print("C-MAPSS already downloaded.")
else:
    print("Downloading C-MAPSS from Kaggle (behrad3d/nasa-cmaps) …")
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", "behrad3d/nasa-cmaps",
             "-p", str(CMAPSS_RAW), "--unzip"],
            capture_output=True, text=True, check=True
        )
        print("Download complete.")
        # flatten any sub-folder
        import shutil as _shutil
        for sub in CMAPSS_RAW.iterdir():
            if sub.is_dir():
                all_moved = True
                for f in sub.iterdir():
                    dest = CMAPSS_RAW / f.name
                    try:
                        _shutil.move(str(f), str(dest))
                    except Exception as move_err:
                        print(f"  WARNING: could not move {f.name}: {move_err}")
                        all_moved = False
                if all_moved:
                    try:
                        sub.rmdir()
                    except OSError as rmdir_err:
                        print(f"  WARNING: could not remove dir {sub.name}: {rmdir_err}")
    except Exception as e:
        print(f"Kaggle download failed: {e}")

        print("\nManual fallback:")
        print("  1. Download CMAPSSData.zip from https://ti.arc.nasa.gov/c/6/")

        print("  2. Extract into  data/raw/cmapss/")

        print("  3. Files needed: train_FD001.txt … train_FD004.txt,")
        print("                   test_FD001.txt  … test_FD004.txt,")
        print("                   RUL_FD001.txt   … RUL_FD004.txt")

files = sorted(CMAPSS_RAW.glob("*.txt"))
print(f"\nC-MAPSS files ({len(files)}):", [f.name for f in files])

In [ ]:
# 2-D  Download Wind SCADA (Kaggle)
from pathlib import Path
import shutil
import subprocess

SCADA_RAW = Path("data/raw/wind_scada")
SCADA_RAW.mkdir(parents=True, exist_ok=True)
sentinel = next(SCADA_RAW.glob("*.csv"), None)

if sentinel:
    print(f"Wind SCADA already downloaded: {sentinel.name}")
else:
    print("Downloading Wind SCADA from Kaggle ...")

    # Kaggle slugs can change over time; try current known slug first.
    candidate_slugs = [
        "berkerisen/wind-turbine-scada-dataset",
        "berkerisen/wind-power-forecasting",
    ]

    download_ok = False
    last_error = ""

    for slug in candidate_slugs:
        print(f"  Trying dataset: {slug}")
        cmd = [
            "kaggle", "datasets", "download",
            "-d", slug,
            "-p", str(SCADA_RAW),
            "--unzip",
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True)

        if proc.returncode == 0:
            print(f"  Download complete from {slug}.")
            download_ok = True
            break

        # Keep short diagnostic output to understand auth/not-found errors.
        err_text = (proc.stderr or proc.stdout or "").strip()
        last_error = err_text
        print(f"  Failed for {slug} (exit code {proc.returncode}).")
        if err_text:
            print("  Kaggle message:")
            for line in err_text.splitlines()[:8]:
                print(f"    {line}")

    if download_ok:
        # Flatten nested folders, if Kaggle extracted into a subdirectory.
        for sub in SCADA_RAW.iterdir():
            if sub.is_dir():
                all_moved = True
                for f in sub.iterdir():
                    dest = SCADA_RAW / f.name
                    try:
                        shutil.move(str(f), str(dest))
                    except Exception as move_err:
                        print(f"  WARNING: could not move {f.name}: {move_err}")
                        all_moved = False
                if all_moved:
                    try:
                        sub.rmdir()
                    except OSError as rmdir_err:
                        print(f"  WARNING: could not remove dir {sub.name}: {rmdir_err}")

        files = sorted(SCADA_RAW.glob("*.csv"))
        if files:
            print(f"CSV files ({len(files)}): {[f.name for f in files]}")
        else:
            print("Download command succeeded, but no CSV files were found.")
            print("Check dataset contents manually in data/raw/wind_scada/.")
    else:
        print("Kaggle download failed for all candidate dataset slugs.")
        if last_error:
            print("Last Kaggle error message:")
            print(last_error[:1200])
        print("Manual: https://www.kaggle.com/datasets/berkerisen/wind-turbine-scada-dataset")

In [ ]:
# 2-E  Download PHM 2010 Milling dataset
# Auto-download uses a Kaggle mirror; official fallback is the PHM Society page.

import shutil
import subprocess
import sys
from pathlib import Path

PHM_RAW = Path("data/raw/phm_milling")
PHM_RAW.mkdir(parents=True, exist_ok=True)

def has_phm_milling_raw(root: Path) -> bool:
    csv_files = list(root.rglob("*.csv"))
    wear_files = [p for p in csv_files if "wear" in p.stem.lower()]
    acquisition_files = [p for p in csv_files if "wear" not in p.stem.lower()]
    return bool(wear_files) and len(acquisition_files) >= 10

def flatten_single_subdir(dest: Path) -> None:
    children = list(dest.iterdir())
    if len(children) != 1 or not children[0].is_dir():
        return
    wrapper = children[0]
    for item in list(wrapper.iterdir()):
        target = dest / item.name
        if target.exists():
            continue
        shutil.move(str(item), str(target))
    try:
        wrapper.rmdir()
    except OSError:
        pass

def kaggle_cmd():
    which = shutil.which("kaggle")
    if which:
        return [which]
    return [sys.executable, "-m", "kaggle.cli"]

if has_phm_milling_raw(PHM_RAW):
    print("PHM Milling already downloaded.")
else:
    success = False
    last_error = ""
    for slug in ["rabahba/phm-data-challenge-2010"]:
        print(f"Trying Kaggle dataset: {slug}")
        proc = subprocess.run(
            kaggle_cmd() + ["datasets", "download", "-d", slug,
                            "-p", str(PHM_RAW), "--unzip"],
            capture_output=True, text=True, check=False,
        )
        if proc.returncode == 0:
            flatten_single_subdir(PHM_RAW)
            if has_phm_milling_raw(PHM_RAW):
                success = True
                print("PHM Milling download complete.")
                break
        last_error = (proc.stderr or proc.stdout or "").strip()

    if not success:
        print("Automated PHM Milling download failed.")
        if last_error:
            print(last_error[:1200])
        print("Manual fallback:")
        print("  https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/")
        print("Expected contents: cutter wear CSVs plus per-cut acquisition CSV files.")

csv_count = len(list(PHM_RAW.rglob("*.csv")))
print(f"CSV files detected: {csv_count}")

In [ ]:
# 2-F  Verify downloads
from pathlib import Path

checks = {
    "C-MAPSS FD001 train": Path("data/raw/cmapss/train_FD001.txt"),
    "C-MAPSS FD001 test":  Path("data/raw/cmapss/test_FD001.txt"),
    "Wind SCADA CSV":      next(Path("data/raw/wind_scada").glob("*.csv"), Path("MISSING")),
    "PHM Milling CSV":     next(Path("data/raw/phm_milling").rglob("*.csv"), Path("MISSING")),
}

all_ok = True
for label, path in checks.items():
    exists = path.exists()
    print(f"  {'✓' if exists else '✗'} {label}: {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll required data found — ready for preprocessing.")
else:
    print("\nSome data is missing. Please complete the download steps above before continuing.")


## 🔧 Section 3 — Data Preprocessing

- **C-MAPSS:** Per-engine chronological split, Kalman imputation, 13 informative sensors  
  Context window = 64 cycles, RUL horizon = 30 cycles  
- **Wind SCADA:** Multivariate CSV → slide windows (512 / 96)  
- **PHM Milling:** stream each raw cut CSV, summarize to cut-level features, then create windows (128 / 16)  

All processed files are saved as `.pt` PyTorch tensors under `data/processed/`.


In [ ]:
# 3-A  Import preprocessing module
from src.data.preprocessing import SCADAPreprocessor
from pathlib import Path

CMAPSS_RAW  = Path("data/raw/cmapss")
SCADA_RAW   = Path("data/raw/wind_scada")
PHM_RAW     = Path("data/raw/phm_milling")
PROC_DIR    = Path("data/processed")


In [ ]:
# 3-B  C-MAPSS — all four subsets
import torch

preprocessor_cmapss = SCADAPreprocessor(
    lookback=CMAPSS_LOOKBACK,
    horizon=CMAPSS_HORIZON,
    train_ratio=0.70,
    val_ratio=0.15,
    normalization="standard",
    seed=SEED,
)

cmapss_data = {}
for subset in CMAPSS_SUBSETS:
    out_dir = PROC_DIR / "cmapss" / subset
    sentinel = out_dir / "processed_data.pt"

    if sentinel.exists():
        print(f"  Loading cached C-MAPSS {subset} …")
        cmapss_data[subset] = preprocessor_cmapss.load_processed(out_dir)
    else:
        print(f"\nPreprocessing C-MAPSS {subset} …")
        data = preprocessor_cmapss.process_cmapss(CMAPSS_RAW, subset=subset)
        preprocessor_cmapss.save_processed(data, out_dir)
        cmapss_data[subset] = data

    d = cmapss_data[subset]
    print(f"  {subset}: train {tuple(d['train_X'].shape)}  "
          f"val {tuple(d['val_X'].shape)}  test {tuple(d['test_X'].shape)}")


In [ ]:
# 3-C  Wind SCADA preprocessing
import pandas as pd
from pathlib import Path

preprocessor_scada = SCADAPreprocessor(
    lookback=LOOKBACK, horizon=HORIZON, train_ratio=0.70,
    val_ratio=0.15, normalization="standard", seed=SEED,
)

SCADA_OUT = PROC_DIR / "wind_scada"
sentinel  = SCADA_OUT / "processed_data.pt"

if sentinel.exists():
    print("Loading cached Wind SCADA …")
    scada_data = preprocessor_scada.load_processed(SCADA_OUT)
else:
    csv_files = sorted(Path("data/raw/wind_scada").glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("No CSV found in data/raw/wind_scada/ — run download step first.")
    csv_path = csv_files[0]
    print(f"Preprocessing Wind SCADA from {csv_path.name} …")

    # Detect timestamp column and drop non-numeric columns
    df_raw = pd.read_csv(csv_path, nrows=5)
    ts_col = next((c for c in df_raw.columns
                   if any(k in c.lower() for k in ["time", "date", "timestamp"])), None)

    scada_data = preprocessor_scada.process_generic_csv(
        csv_path,
        timestamp_col=ts_col,
        task="forecasting",
    )
    preprocessor_scada.save_processed(scada_data, SCADA_OUT)

d = scada_data
print(f"Wind SCADA: train {tuple(d['train_X'].shape)}  "
      f"val {tuple(d['val_X'].shape)}  test {tuple(d['test_X'].shape)}")


In [ ]:
# 3-D  PHM Milling — streaming cut summaries → preprocessing
import importlib
import importlib.util
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from src.data.preprocessing import SCADAPreprocessor

scripts_dir = Path("scripts").resolve()
module_path = scripts_dir / "step_02_preprocess.py"

if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))


def _load_repo_preprocessor():
    if not module_path.exists():
        return None

    try:
        importlib.invalidate_caches()
        module_name = "step_02_preprocess"
        if module_name in sys.modules:
            del sys.modules[module_name]

        spec = importlib.util.spec_from_file_location(module_name, module_path)
        if spec is None or spec.loader is None:
            return None

        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return getattr(module, "preprocess_phm_milling", None)
    except Exception as exc:
        print(f"Could not import preprocess_phm_milling from {module_path.name}: {exc}")
        return None


preprocess_phm_milling = _load_repo_preprocessor()

if preprocess_phm_milling is None:
    print("Using notebook-local PHM preprocessing fallback.")

    phm_raw_dir = Path(globals().get("PHM_RAW", "data/raw/phm_milling"))
    proc_dir_root = Path(globals().get("PROC_DIR", "data/processed"))
    lookback = int(globals().get("PHM_MILLING_LOOKBACK", 128))
    horizon = int(globals().get("PHM_MILLING_HORIZON", 16))
    seed = int(globals().get("SEED", 42))
    chunk_rows = 100_000

    phm_sensor_names = [
        "force_x",
        "force_y",
        "force_z",
        "vibration_x",
        "vibration_y",
        "vibration_z",
        "ae_rms",
    ]
    cutter_re = re.compile(r"(c\d+)", re.IGNORECASE)
    digit_re = re.compile(r"(\d+)")

    def _cat_or_empty(arrays, tail_shape):
        if arrays:
            return torch.FloatTensor(np.concatenate(arrays, axis=0))
        return torch.empty((0, *tail_shape), dtype=torch.float32)

    def _stack_or_empty(arrays, tail_shape):
        if arrays:
            return np.asarray(arrays, dtype=np.float32)
        return np.empty((0, *tail_shape), dtype=np.float32)

    def _infer_cutter_id(path: Path):
        for candidate in [path.stem, path.parent.name, *[parent.name for parent in path.parents[:3]]]:
            match = cutter_re.search(candidate)
            if match:
                return match.group(1).lower()
        return None

    def _cut_sort_key(path: Path):
        digits = digit_re.findall(path.stem)
        if digits:
            return (0, int(digits[-1]), path.name.lower())
        return (1, path.name.lower())

    def _iter_numeric_chunks(csv_path: Path):
        read_attempts = [
            {"header": None, "chunksize": chunk_rows},
            {"header": None, "sep": r"[,\s]+", "engine": "python", "chunksize": chunk_rows},
        ]

        last_error = None
        for kwargs in read_attempts:
            try:
                reader = pd.read_csv(csv_path, **kwargs)
                first = next(reader)
            except StopIteration:
                return
            except Exception as exc:
                last_error = exc
                continue

            first = first.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
            if first.shape[1] <= 1 and "sep" not in kwargs:
                continue

            yield first
            for chunk in reader:
                numeric = chunk.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
                if not numeric.empty:
                    yield numeric
            return

        if last_error is not None:
            raise last_error
        raise ValueError(f"Could not parse numeric cut file: {csv_path}")

    def _summarize_cut_file(csv_path: Path):
        count = 0
        sum_vec = None
        sumsq_vec = None
        min_vec = None
        max_vec = None

        for chunk in _iter_numeric_chunks(csv_path):
            arr = chunk.to_numpy(dtype=np.float64, copy=False)
            if arr.ndim == 1:
                arr = arr[:, None]
            if arr.shape[1] < len(phm_sensor_names):
                continue

            arr = arr[:, :len(phm_sensor_names)]
            arr = arr[np.isfinite(arr).all(axis=1)]
            if arr.size == 0:
                continue

            if sum_vec is None:
                sum_vec = arr.sum(axis=0)
                sumsq_vec = np.square(arr).sum(axis=0)
                min_vec = arr.min(axis=0)
                max_vec = arr.max(axis=0)
            else:
                sum_vec += arr.sum(axis=0)
                sumsq_vec += np.square(arr).sum(axis=0)
                min_vec = np.minimum(min_vec, arr.min(axis=0))
                max_vec = np.maximum(max_vec, arr.max(axis=0))
            count += arr.shape[0]

        if count == 0 or sum_vec is None:
            return None

        mean_vec = sum_vec / count
        var_vec = np.maximum(sumsq_vec / count - np.square(mean_vec), 0.0)
        std_vec = np.sqrt(var_vec)
        rms_vec = np.sqrt(sumsq_vec / count)

        features = {}
        for idx, sensor_name in enumerate(phm_sensor_names):
            features[f"{sensor_name}_mean"] = float(mean_vec[idx])
            features[f"{sensor_name}_std"] = float(std_vec[idx])
            features[f"{sensor_name}_min"] = float(min_vec[idx])
            features[f"{sensor_name}_max"] = float(max_vec[idx])
            features[f"{sensor_name}_rms"] = float(rms_vec[idx])
        return features

    def _discover_phm_cutters(root: Path):
        groups = {}
        for csv_path in sorted(root.rglob("*.csv")):
            cutter_id = _infer_cutter_id(csv_path)
            if cutter_id is None:
                continue
            group = groups.setdefault(cutter_id, {"cuts": [], "wear": None})
            if "wear" in csv_path.stem.lower():
                group["wear"] = csv_path
            else:
                group["cuts"].append(csv_path)
        return {cutter_id: group for cutter_id, group in groups.items() if group["cuts"]}

    def _split_forecasting_trajectory(preprocessor: SCADAPreprocessor, data: np.ndarray):
        splits = preprocessor._chronological_split(data)
        train_n, val_n, test_n = preprocessor._normalize(
            splits["train"], splits["val"], splits["test"]
        )

        full_series = np.concatenate([train_n, val_n, test_n], axis=0)
        train_end = len(train_n)
        val_end = train_end + len(val_n)

        X_train, y_train = [], []
        X_val, y_val = [], []
        X_test, y_test = [], []

        max_start = len(full_series) - preprocessor.lookback - preprocessor.horizon + 1
        for start in range(max(0, max_start)):
            X_window = full_series[start:start + preprocessor.lookback]
            y_window = full_series[
                start + preprocessor.lookback:start + preprocessor.lookback + preprocessor.horizon
            ]
            target_end = start + preprocessor.lookback + preprocessor.horizon - 1

            if target_end < train_end:
                X_train.append(X_window)
                y_train.append(y_window)
            elif target_end < val_end:
                X_val.append(X_window)
                y_val.append(y_window)
            else:
                X_test.append(X_window)
                y_test.append(y_window)

        feature_dim = data.shape[1]
        return (
            _stack_or_empty(X_train, (preprocessor.lookback, feature_dim)),
            _stack_or_empty(y_train, (preprocessor.horizon, feature_dim)),
            _stack_or_empty(X_val, (preprocessor.lookback, feature_dim)),
            _stack_or_empty(y_val, (preprocessor.horizon, feature_dim)),
            _stack_or_empty(X_test, (preprocessor.lookback, feature_dim)),
            _stack_or_empty(y_test, (preprocessor.horizon, feature_dim)),
        )

    def preprocess_phm_milling():
        out_dir = proc_dir_root / "phm_milling"
        sentinel = out_dir / "processed_data.pt"

        preprocessor = SCADAPreprocessor(
            lookback=lookback,
            horizon=horizon,
            train_ratio=0.70,
            val_ratio=0.15,
            normalization="standard",
            seed=seed,
        )

        if sentinel.exists():
            print("Loading cached PHM Milling …")
            return preprocessor.load_processed(out_dir)

        cutter_groups = _discover_phm_cutters(phm_raw_dir)
        if not cutter_groups:
            raise FileNotFoundError(
                "No PHM Milling cutter CSV files were found. Run the download cells first."
            )

        all_X_train, all_y_train = [], []
        all_X_val, all_y_val = [], []
        all_X_test, all_y_test = [], []
        feature_tables = []
        feature_dim = None

        for cutter_id, group in sorted(cutter_groups.items()):
            cut_files = sorted(group["cuts"], key=_cut_sort_key)
            print(f"Summarizing {cutter_id}: {len(cut_files)} cut files")

            rows = []
            for cut_number, cut_file in enumerate(cut_files, start=1):
                features = _summarize_cut_file(cut_file)
                if features is None:
                    continue
                features["cut_index"] = cut_number
                rows.append(features)

            if len(rows) < lookback + horizon:
                print(
                    f"  Skipping {cutter_id}: only {len(rows)} cut summaries, "
                    f"need at least {lookback + horizon}."
                )
                continue

            feature_df = pd.DataFrame(rows).sort_values("cut_index").reset_index(drop=True)
            feature_tables.append(feature_df.assign(cutter_id=cutter_id))

            data = feature_df.drop(columns=["cut_index"]).to_numpy(dtype=np.float32)
            feature_dim = data.shape[1]
            X_tr, y_tr, X_va, y_va, X_te, y_te = _split_forecasting_trajectory(preprocessor, data)

            if len(X_tr):
                all_X_train.append(X_tr)
                all_y_train.append(y_tr)
            if len(X_va):
                all_X_val.append(X_va)
                all_y_val.append(y_va)
            if len(X_te):
                all_X_test.append(X_te)
                all_y_test.append(y_te)

        if feature_dim is None or not feature_tables:
            raise RuntimeError(
                "PHM Milling preprocessing did not produce any usable cut-level sequences."
            )

        result = {
            "train_X": _cat_or_empty(all_X_train, (lookback, feature_dim)),
            "train_y": _cat_or_empty(all_y_train, (horizon, feature_dim)),
            "val_X": _cat_or_empty(all_X_val, (lookback, feature_dim)),
            "val_y": _cat_or_empty(all_y_val, (horizon, feature_dim)),
            "test_X": _cat_or_empty(all_X_test, (lookback, feature_dim)),
            "test_y": _cat_or_empty(all_y_test, (horizon, feature_dim)),
            "task": "forecasting",
            "dataset": "phm_milling",
            "num_channels": feature_dim,
            "lookback": lookback,
            "horizon": horizon,
            "cutters": sorted(cutter_groups.keys()),
        }

        out_dir.mkdir(parents=True, exist_ok=True)
        pd.concat(feature_tables, ignore_index=True).to_csv(out_dir / "cut_level_features.csv", index=False)
        preprocessor.save_processed(result, out_dir)
        return result

phm_data = preprocess_phm_milling()
d = phm_data
print(f"PHM Milling: train {tuple(d['train_X'].shape)}  "
      f"val {tuple(d['val_X'].shape)}  test {tuple(d['test_X'].shape)}")

In [ ]:
# 3-E  Checkpoint preprocessed data to Drive
import shutil
from pathlib import Path

if DRIVE_AVAILABLE and DRIVE_RESULTS_DIR:
    drive_proc = Path(DRIVE_RESULTS_DIR).parent / "processed"
    drive_proc.mkdir(parents=True, exist_ok=True)
    for dataset in ["cmapss", "wind_scada", "phm_milling"]:
        src = Path(f"data/processed/{dataset}")
        dst = drive_proc / dataset
        if src.exists():
            shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
            print(f"  Checkpointed data/processed/{dataset} → Drive")
else:
    print("Drive not available — data stays local.")


## 🔬 Section 4 — Zero-Shot Experiments

All four models are evaluated **without any task-specific training**.  
PatchTST is trained on the training split first (it is a supervised baseline).

Each run saves a JSON result file under `results/zero_shot/`.


In [ ]:
# 4-A  Import model wrappers and metric utilities
from src.models import get_model
from src.evaluation.metrics import compute_all_metrics
import torch, numpy as np, json
from pathlib import Path
from datetime import datetime

RES_ZS = Path("results/zero_shot")
RES_ZS.mkdir(parents=True, exist_ok=True)

def load_dataset(name: str, subset: str = None):
    '''Return (X_train, y_train, X_test, y_test) as numpy arrays.'''
    if name == "cmapss":
        key = subset or "FD001"
        d = cmapss_data[key]
    elif name == "wind_scada":
        d = scada_data
    elif name == "phm_milling":
        d = phm_data
    else:
        raise ValueError(f"Unknown dataset: {name}")
    to_np = lambda t: t.numpy() if isinstance(t, torch.Tensor) else t
    return (to_np(d["train_X"]), to_np(d["train_y"]),
            to_np(d["test_X"]),  to_np(d["test_y"]))


In [ ]:
# 4-B  Run zero-shot experiments
import gc
import traceback

ZERO_SHOT_BATCH_SIZES = globals().get(
    "ZERO_SHOT_BATCH_SIZES",
    {"moment": 4, "chronos": 8, "lag_llama": 16, "patchtst": 16},
)
ZERO_SHOT_MAX_TEST_SAMPLES = globals().get("ZERO_SHOT_MAX_TEST_SAMPLES", None)

def clear_runtime_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except RuntimeError:
            pass

def is_cuda_oom(exc: Exception) -> bool:
    if isinstance(exc, torch.cuda.OutOfMemoryError):
        return True
    return "out of memory" in str(exc).lower()

def batched_zero_shot_predict(model_name: str, model, X_test, horizon: int):
    if ZERO_SHOT_MAX_TEST_SAMPLES is not None and len(X_test) > ZERO_SHOT_MAX_TEST_SAMPLES:
        X_test = X_test[:ZERO_SHOT_MAX_TEST_SAMPLES]

    if len(X_test) == 0:
        raise ValueError("X_test is empty; nothing to evaluate.")

    batch_size = min(ZERO_SHOT_BATCH_SIZES.get(model_name, 8), len(X_test))
    pred_chunks = []
    start = 0

    while start < len(X_test):
        stop = min(start + batch_size, len(X_test))
        batch = torch.FloatTensor(X_test[start:stop])
        try:
            results = model.zero_shot(batch, horizon=horizon)
            pred_chunks.append(results["predictions"])
            start = stop
        except Exception as exc:
            if not is_cuda_oom(exc) or batch_size == 1:
                raise
            clear_runtime_memory()
            batch_size = max(1, batch_size // 2)
            print(f"  CUDA OOM with batch size {stop - start}; retrying with batch size {batch_size} …")
        finally:
            del batch
            clear_runtime_memory()

    return np.concatenate(pred_chunks, axis=0)

def run_zero_shot(model_name: str, dataset_name: str,
                  X_train, y_train, X_test, y_test,
                  horizon: int) -> dict:
    model = None
    try:
        clear_runtime_memory()
        model = get_model(model_name, device=DEVICE)

        if model_name == "patchtst":
            print(f"  PatchTST: fitting on training split first …")
            model.fit(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
        else:
            model.load_model()

        preds = batched_zero_shot_predict(model_name, model, X_test, horizon)

        if ZERO_SHOT_MAX_TEST_SAMPLES is not None and len(y_test) > len(preds):
            y = y_test[:len(preds)]
        else:
            y = y_test

        if y.ndim == 1:
            preds_flat = preds.reshape(preds.shape[0], -1).mean(axis=1)
            y_flat     = y
        else:
            min_h = min(preds.shape[1], y.shape[1] if y.ndim > 1 else horizon)
            preds_flat = preds[:, :min_h].flatten()
            y_flat     = y[:, :min_h].flatten() if y.ndim > 1 else y.flatten()

        metrics = {
            "mae":  float(np.mean(np.abs(y_flat - preds_flat))),
            "rmse": float(np.sqrt(np.mean((y_flat - preds_flat) ** 2))),
            "mape": float(np.mean(np.abs((y_flat - preds_flat) /
                                     (np.abs(y_flat) + 1e-8))) * 100),
        }
        return metrics
    finally:
        if model is not None:
            del model
        clear_runtime_memory()

if not RUN_ZERO_SHOT:
    print("RUN_ZERO_SHOT=False — skipping section 4.")
else:
    zero_shot_results = {}

    # Dataset configs: (name, subset_or_None, horizon)
    dataset_cfgs = [
        ("cmapss",      "FD001", CMAPSS_HORIZON),
        ("wind_scada",  None,    HORIZON),
        ("phm_milling", None,    PHM_MILLING_HORIZON),
    ]

    for ds_name, subset, h in dataset_cfgs:
        X_tr, y_tr, X_te, y_te = load_dataset(ds_name, subset)
        ds_label = f"{ds_name}/{subset}" if subset else ds_name

        for model_name in MODELS_ZERO_SHOT:
            key = f"{model_name}_{ds_label.replace('/', '_')}"
            json_path = RES_ZS / f"{key}.json"

            if json_path.exists():
                print(f"  [cached] {key}")
                with open(json_path) as f:
                    zero_shot_results[key] = json.load(f)
                continue

            print(f"\n{'='*60}")
            print(f"  Zero-shot: {model_name} on {ds_label}")
            print(f"{'='*60}")
            try:
                metrics = run_zero_shot(model_name, ds_name,
                                        X_tr, y_tr, X_te, y_te, h)
                record = {
                    "model": model_name, "dataset": ds_label,
                    "scenario": "zero_shot", "metrics": metrics,
                    "timestamp": datetime.now().isoformat(),
                }
                zero_shot_results[key] = record
                with open(json_path, "w") as f:
                    json.dump(record, f, indent=2)
                print(f"  → {metrics}")
            except Exception as e:
                print(f"  ERROR: {e}")
                print(traceback.format_exc(limit=8))
                zero_shot_results[key] = {"model": model_name,
                                          "dataset": ds_label, "error": str(e)}

    completed = sum(1 for record in zero_shot_results.values() if "error" not in record)
    failed = sum(1 for record in zero_shot_results.values() if "error" in record)
    print(f"\nZero-shot done. {completed} successful run(s), {failed} failed run(s). Results saved to {RES_ZS}")

## 🎯 Section 5 — Few-Shot LoRA Experiments

Only **MOMENT** and **Lag-Llama** support LoRA fine-tuning.  
Each model is fine-tuned using **1 %** of the training data (≥ 32 samples).  
LoRA config: `r=16`, `α=32`, `10 epochs`.


In [ ]:
# 5-A  Run few-shot LoRA experiments

RES_FS = Path("results/few_shot")
RES_FS.mkdir(parents=True, exist_ok=True)

def run_few_shot(model_name: str, dataset_name: str,
                 X_train, y_train, X_test, y_test,
                 horizon: int, train_ratio: float = TRAIN_RATIO) -> dict:
    n_few = max(int(len(X_train) * train_ratio), 32)
    X_few = torch.FloatTensor(X_train[:n_few])
    y_few = torch.FloatTensor(y_train[:n_few])

    model = get_model(model_name, device=DEVICE)
    if not getattr(model, "supports_few_shot", False):
        raise NotImplementedError(
            f"{model_name} few-shot adaptation is not implemented in this repository."
        )
    model.load_model()
    model.few_shot_adapt(X_few, y_few,
                         epochs=LORA_EPOCHS, lr=LORA_LR,
                         lora_r=LORA_R, lora_alpha=LORA_ALPHA)

    results = model.predict(torch.FloatTensor(X_test), horizon=horizon)
    preds   = results["predictions"]

    y = y_test
    if y.ndim == 1:
        preds_flat = preds.reshape(preds.shape[0], -1).mean(axis=1)
        y_flat     = y
    else:
        min_h      = min(preds.shape[1], y.shape[1] if y.ndim > 1 else horizon)
        preds_flat = preds[:, :min_h].flatten()
        y_flat     = y[:, :min_h].flatten()

    metrics = {
        "mae":         float(np.mean(np.abs(y_flat - preds_flat))),
        "rmse":        float(np.sqrt(np.mean((y_flat - preds_flat) ** 2))),
        "mape":        float(np.mean(np.abs((y_flat - preds_flat) /
                                             (np.abs(y_flat) + 1e-8))) * 100),
        "train_samples": n_few,
    }

    del model
    torch.cuda.empty_cache()
    return metrics


if not RUN_FEW_SHOT:
    print("RUN_FEW_SHOT=False — skipping section 5.")
else:
    few_shot_results = {}

    dataset_cfgs_fs = [
        ("cmapss",      "FD001", CMAPSS_HORIZON),
        ("wind_scada",  None,    HORIZON),
        ("phm_milling", None,    PHM_MILLING_HORIZON),
    ]

    for ds_name, subset, h in dataset_cfgs_fs:
        X_tr, y_tr, X_te, y_te = load_dataset(ds_name, subset)
        ds_label = f"{ds_name}/{subset}" if subset else ds_name

        for model_name in MODELS_FEW_SHOT:
            key       = f"{model_name}_{ds_label.replace('/', '_')}_few_shot"
            json_path = RES_FS / f"{key}.json"

            if json_path.exists():
                print(f"  [cached] {key}")
                with open(json_path) as f:
                    few_shot_results[key] = json.load(f)
                continue

            print(f"\n{'='*60}")
            print(f"  Few-shot LoRA: {model_name} on {ds_label}")
            print(f"{'='*60}")
            try:
                metrics = run_few_shot(model_name, ds_name,
                                       X_tr, y_tr, X_te, y_te, h)
                record = {
                    "model": model_name, "dataset": ds_label,
                    "scenario": "few_shot", "metrics": metrics,
                    "timestamp": datetime.now().isoformat(),
                }
                few_shot_results[key] = record
                with open(json_path, "w") as f:
                    json.dump(record, f, indent=2)
                print(f"  → {metrics}")
            except Exception as e:
                print(f"  ERROR: {e}")
                traceback.print_exc()
                few_shot_results[key] = {"model": model_name,
                                         "dataset": ds_label, "error": str(e)}

    print(f"\nFew-shot done. {len(few_shot_results)} run(s) saved to {RES_FS}")


## 🔄 Section 6 — Cross-Condition Transfer (C-MAPSS)

Train condition: **FD001** (one fault mode, moderate fan degradation)  
Transfer targets: **FD002**, **FD003**, **FD004** (more fan + HPC faults, higher variability)

For zero-shot TSFMs, no retraining is needed — the model simply runs inference on
the target condition directly (domain-agnostic feature extraction).


In [ ]:
# 6-A  Cross-condition experiments  (FD001 → FD002/003/004)
#      ALSO runs FD001 in-domain evaluation — needed for Table 5 FD001 baseline column

RES_CC = Path("results/cross_condition")
RES_CC.mkdir(parents=True, exist_ok=True)

SOURCE_SUBSET  = "FD001"
TARGET_SUBSETS = [s for s in CMAPSS_SUBSETS if s != SOURCE_SUBSET]

if not RUN_CROSS_CONDITION:
    print("RUN_CROSS_CONDITION=False — skipping section 6.")
else:
    cross_cond_results = {}

    X_src_tr_full, y_src_tr_full, X_src_te, y_src_te = load_dataset("cmapss", SOURCE_SUBSET)

    # ── Step A: FD001 in-domain baseline (Table 5 FD001 column) ──────────────
    print("\n── FD001 in-domain baseline (Table 5 FD001 column) ──")
    for model_name in MODELS_ZERO_SHOT:
        key       = f"{model_name}_FD001_to_FD001"
        json_path = RES_CC / f"{key}.json"

        if json_path.exists():
            print(f"  [cached] {key}")
            with open(json_path) as f:
                cross_cond_results[key] = json.load(f)
            continue

        print(f"\n  In-domain: {model_name} on FD001")
        try:
            model = get_model(model_name, device=DEVICE)
            if model_name == "patchtst":
                model.fit(torch.FloatTensor(X_src_tr_full),
                          torch.FloatTensor(y_src_tr_full))
            else:
                model.load_model()

            results = model.predict(torch.FloatTensor(X_src_te), horizon=CMAPSS_HORIZON)
            preds   = results["predictions"]

            y = y_src_te
            if y.ndim == 1:
                preds_flat = preds.reshape(preds.shape[0], -1).mean(axis=1)
                y_flat     = y
            else:
                min_h      = min(preds.shape[1], y.shape[1])
                preds_flat = preds[:, :min_h].flatten()
                y_flat     = y[:, :min_h].flatten()

            metrics = {
                "mae":  float(np.mean(np.abs(y_flat - preds_flat))),
                "rmse": float(np.sqrt(np.mean((y_flat - preds_flat) ** 2))),
            }
            record = {
                "model": model_name,
                "source": SOURCE_SUBSET, "target": SOURCE_SUBSET,
                "scenario": "cross_condition", "metrics": metrics,
                "timestamp": datetime.now().isoformat(),
            }
            cross_cond_results[key] = record
            with open(json_path, "w") as f:
                json.dump(record, f, indent=2)
            print(f"    → {metrics}")

            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception as e:
            print(f"    ERROR: {e}")
            traceback.print_exc()

    # ── Step B: FD001 → FD002 / FD003 / FD004 ────────────────────────────────
    for target in TARGET_SUBSETS:
        _, _, X_tgt_te, y_tgt_te = load_dataset("cmapss", target)

        for model_name in MODELS_ZERO_SHOT:
            key       = f"{model_name}_FD001_to_{target}"
            json_path = RES_CC / f"{key}.json"

            if json_path.exists():
                print(f"  [cached] {key}")
                with open(json_path) as f:
                    cross_cond_results[key] = json.load(f)
                continue

            print(f"\n  Cross-condition: {model_name}  FD001 → {target}")
            try:
                model = get_model(model_name, device=DEVICE)
                if model_name == "patchtst":
                    model.fit(torch.FloatTensor(X_src_tr_full),
                              torch.FloatTensor(y_src_tr_full))
                else:
                    model.load_model()

                results = model.predict(torch.FloatTensor(X_tgt_te), horizon=CMAPSS_HORIZON)
                preds   = results["predictions"]

                y = y_tgt_te
                if y.ndim == 1:
                    preds_flat = preds.reshape(preds.shape[0], -1).mean(axis=1)
                    y_flat     = y
                else:
                    min_h      = min(preds.shape[1], y.shape[1])
                    preds_flat = preds[:, :min_h].flatten()
                    y_flat     = y[:, :min_h].flatten()

                metrics = {
                    "mae":  float(np.mean(np.abs(y_flat - preds_flat))),
                    "rmse": float(np.sqrt(np.mean((y_flat - preds_flat) ** 2))),
                }
                record = {
                    "model": model_name,
                    "source": SOURCE_SUBSET, "target": target,
                    "scenario": "cross_condition", "metrics": metrics,
                    "timestamp": datetime.now().isoformat(),
                }
                cross_cond_results[key] = record
                with open(json_path, "w") as f:
                    json.dump(record, f, indent=2)
                print(f"    → {metrics}")

                del model
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            except Exception as e:
                print(f"    ERROR: {e}")
                traceback.print_exc()
                cross_cond_results[key] = {"model": model_name,
                                           "source": SOURCE_SUBSET,
                                           "target": target, "error": str(e)}

    print(f"\nCross-condition done. {len(cross_cond_results)} run(s) saved to {RES_CC}")


## 📊 Section 7 — Results Aggregation & Tables

In [ ]:
# 7-A  Load all results into DataFrames
import pandas as pd, json
from pathlib import Path

def load_results_dir(res_dir: Path) -> pd.DataFrame:
    rows = []
    for f in sorted(res_dir.glob("*.json")):
        with open(f) as fp:
            r = json.load(fp)
        if "error" in r:
            continue
        m = r.get("metrics", {})
        row = {
            "scenario":  r.get("scenario", res_dir.name),
            "model":     r.get("model"),
            "dataset":   r.get("dataset", r.get("target")),
            "mae":       m.get("mae"),
            "rmse":      m.get("rmse"),
            "mape":      m.get("mape"),
        }
        if "source" in r:
            row["source"]  = r["source"]
            row["target"]  = r["target"]
        rows.append(row)
    return pd.DataFrame(rows)

df_zs = load_results_dir(Path("results/zero_shot"))
df_fs = load_results_dir(Path("results/few_shot"))
df_cc = load_results_dir(Path("results/cross_condition"))
df_all = pd.concat([df_zs, df_fs, df_cc], ignore_index=True)

print(f"Total result records: {len(df_all)}")
print(df_all.head(20).to_string(index=False))


In [ ]:
# 7-B  Zero-shot pivot table (MAE)
if not df_zs.empty:
    pivot_zs = df_zs.pivot_table(
        index="model", columns="dataset", values="mae", aggfunc="mean"
    )
    print("\n=== Zero-Shot MAE ===")
    print(pivot_zs.round(4).to_string())
else:
    print("No zero-shot results yet.")


In [ ]:
# 7-C  Few-shot pivot table (MAE)
if not df_fs.empty:
    pivot_fs = df_fs.pivot_table(
        index="model", columns="dataset", values="mae", aggfunc="mean"
    )
    print("\n=== Few-Shot (LoRA 1%) MAE ===")
    print(pivot_fs.round(4).to_string())
else:
    print("No few-shot results yet.")


In [ ]:
# 7-D  Cross-condition table (MAE, FD001 → FD00x)
if not df_cc.empty:
    pivot_cc = df_cc.pivot_table(
        index="model", columns="target", values="mae", aggfunc="mean"
    )
    print("\n=== Cross-Condition MAE (source: FD001) ===")
    print(pivot_cc.round(4).to_string())
else:
    print("No cross-condition results yet.")


## ⏱️ Section 8 — Inference Benchmarking & Results Export

Profiles **inference latency**, **peak GPU memory**, and **parameter count** for each model.  
Then exports all aggregated results as clean **CSV files** under `results/tables/` for later use in article writing.

In [ ]:
# 8-A  Inference latency & GPU memory profiling
import time, json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from src.models import get_model

PROFILE_DIR = Path("results/tables")
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

# Fixed profiling batch: 32 C-MAPSS test samples
_, _, X_prof_np, _ = load_dataset("cmapss", "FD001")
X_prof = torch.FloatTensor(X_prof_np[:32])

WARMUP_RUNS = 2
TIMING_RUNS = 5

profile_results = {}

for model_name in MODELS_ZERO_SHOT:
    print(f"\nProfiling {model_name} ...")
    cached_path = PROFILE_DIR / f"profile_{model_name}.json"
    if cached_path.exists():
        with open(cached_path) as f:
            profile_results[model_name] = json.load(f)
        print(f"  [cached]")
        continue
    try:
        model = get_model(model_name, device=DEVICE)
        if model_name == "patchtst":
            X_tr_p, y_tr_p, *_ = load_dataset("cmapss", "FD001")
            model.fit(torch.FloatTensor(X_tr_p[:64]), torch.FloatTensor(y_tr_p[:64]))
        else:
            model.load_model()

        _inner = getattr(model, "model", None)
        n_params = sum(p.numel() for p in _inner.parameters()) if _inner is not None else 0

        # Warmup
        for _ in range(WARMUP_RUNS):
            _ = model.predict(X_prof, horizon=CMAPSS_HORIZON)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        # Timed runs
        t0 = time.perf_counter()
        for _ in range(TIMING_RUNS):
            _ = model.predict(X_prof, horizon=CMAPSS_HORIZON)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - t0) / TIMING_RUNS * 1000

        peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6
                       if torch.cuda.is_available() else float("nan"))

        rec = {"model": model_name,
               "params_M":    round(n_params / 1e6, 1),
               "latency_ms":  round(elapsed_ms, 1),
               "peak_gpu_mb": round(peak_mem_mb, 0),
               "batch_size":  int(X_prof.shape[0])}
        profile_results[model_name] = rec
        with open(cached_path, "w") as f:
            json.dump(rec, f, indent=2)
        print(f"  params: {rec['params_M']} M  |  latency: {rec['latency_ms']} ms  "
              f"|  peak GPU: {rec['peak_gpu_mb']} MB")

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"  ERROR: {e}")
        profile_results[model_name] = {"model": model_name, "error": str(e)}

# Save efficiency results as CSV
eff_rows = [v for v in profile_results.values() if "error" not in v]
df_eff = pd.DataFrame(eff_rows)
df_eff.to_csv(PROFILE_DIR / "efficiency_summary.csv", index=False)

print("\n=== Deployment Efficiency ===")
print(f"{'Model':<12} {'Params (M)':>11} {'Latency (ms)':>13} {'Peak GPU (MB)':>14}")
print("-" * 55)
for m in MODELS_ZERO_SHOT:
    r2 = profile_results.get(m, {})
    if not r2 or "error" in r2:
        print(f"  {m:<12}  (error)")
    else:
        print(f"  {m:<12} {r2['params_M']:>10.1f} "
              f" {r2['latency_ms']:>12.1f}  {r2['peak_gpu_mb']:>13.0f}")
print(f"\nSaved → results/tables/efficiency_summary.csv")

In [ ]:
# 8-B  Export all aggregated results as CSV files
#      These CSVs contain everything needed to build article tables later.
import pandas as pd
from pathlib import Path

TABLES_DIR = Path("results/tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Zero-shot summary (model × dataset × metrics) ──────────────────────
if not df_zs.empty:
    # Pivot: rows=model, columns=dataset, values=mae
    pivot_zs_mae = df_zs.pivot_table(index="model", columns="dataset",
                                      values="mae", aggfunc="mean")
    pivot_zs_mae["mean"] = pivot_zs_mae.mean(axis=1)
    pivot_zs_mae.to_csv(TABLES_DIR / "zero_shot_mae.csv")

    # Full detail (all metrics)
    df_zs.to_csv(TABLES_DIR / "zero_shot_full.csv", index=False)
    print(f"Zero-shot: {len(df_zs)} runs → zero_shot_mae.csv + zero_shot_full.csv")
    print(pivot_zs_mae.round(4).to_string())
else:
    print("No zero-shot results to export.")

# ── 2. Few-shot summary ───────────────────────────────────────────────────
if not df_fs.empty:
    pivot_fs_mae = df_fs.pivot_table(index="model", columns="dataset",
                                      values="mae", aggfunc="mean")
    pivot_fs_mae["mean"] = pivot_fs_mae.mean(axis=1)
    pivot_fs_mae.to_csv(TABLES_DIR / "few_shot_mae.csv")
    df_fs.to_csv(TABLES_DIR / "few_shot_full.csv", index=False)
    print(f"\nFew-shot: {len(df_fs)} runs → few_shot_mae.csv + few_shot_full.csv")
    print(pivot_fs_mae.round(4).to_string())
else:
    print("No few-shot results to export.")

# ── 3. Zero-shot vs Few-shot comparison (for LoRA-capable models) ─────────
if not df_zs.empty and not df_fs.empty:
    rows_cmp = []
    for m in MODELS_FEW_SHOT:
        for ds in df_zs["dataset"].unique():
            zs_val = df_zs[(df_zs["model"] == m) & (df_zs["dataset"] == ds)]["mae"].mean()
            fs_val = df_fs[(df_fs["model"] == m) & (df_fs["dataset"] == ds)]["mae"].mean()
            if pd.notna(zs_val) or pd.notna(fs_val):
                improvement = ((zs_val - fs_val) / zs_val * 100) if pd.notna(zs_val) and pd.notna(fs_val) and zs_val > 0 else None
                rows_cmp.append({"model": m, "dataset": ds,
                                 "zero_shot_mae": zs_val, "few_shot_mae": fs_val,
                                 "improvement_pct": improvement})
    df_cmp = pd.DataFrame(rows_cmp)
    df_cmp.to_csv(TABLES_DIR / "zs_vs_fs_comparison.csv", index=False)
    print(f"\nZS vs FS comparison → zs_vs_fs_comparison.csv")
    print(df_cmp.round(4).to_string(index=False))

# ── 4. Cross-condition summary ────────────────────────────────────────────
if not df_cc.empty:
    pivot_cc_mae = df_cc.pivot_table(index="model", columns="target",
                                      values="mae", aggfunc="mean")
    # Reorder columns: FD001, FD002, FD003, FD004
    col_order = [c for c in ["FD001", "FD002", "FD003", "FD004"] if c in pivot_cc_mae.columns]
    pivot_cc_mae = pivot_cc_mae[col_order]
    pivot_cc_mae.to_csv(TABLES_DIR / "cross_condition_mae.csv")
    df_cc.to_csv(TABLES_DIR / "cross_condition_full.csv", index=False)
    print(f"\nCross-condition: {len(df_cc)} runs → cross_condition_mae.csv + cross_condition_full.csv")
    print(pivot_cc_mae.round(4).to_string())
else:
    print("No cross-condition results to export.")

# ── 5. Combined master CSV ────────────────────────────────────────────────
df_all.to_csv(TABLES_DIR / "all_results.csv", index=False)

print(f"\n{'='*60}")
print(f"All CSV files saved to: results/tables/")
for f in sorted(TABLES_DIR.glob("*.csv")):
    print(f"  {f.name}")
print(f"{'='*60}")

## 📈 Section 9 — Generate Figures

In [ ]:
# 9-A  Bar chart — Zero-shot MAE by model × dataset
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 150})

FIGS_DIR = Path("results/figures")
FIGS_DIR.mkdir(parents=True, exist_ok=True)

if not df_zs.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    datasets_plot = df_zs["dataset"].unique()
    models_plot   = [m for m in MODELS_ZERO_SHOT if m in df_zs["model"].values]

    x = range(len(models_plot))
    width = 0.8 / max(len(datasets_plot), 1)
    colors = plt.cm.tab10.colors

    for i, ds in enumerate(datasets_plot):
        vals = [
            df_zs[(df_zs["model"] == m) & (df_zs["dataset"] == ds)]["mae"].mean()
            for m in models_plot
        ]
        offset = (i - len(datasets_plot) / 2 + 0.5) * width
        bars = ax.bar([xi + offset for xi in x], vals, width,
                      label=ds, color=colors[i % len(colors)], alpha=0.85)

    ax.set_xticks(list(x))
    ax.set_xticklabels(models_plot, rotation=15, ha="right")
    ax.set_ylabel("MAE")
    ax.set_title("Zero-Shot MAE by Model and Dataset")
    ax.legend(title="Dataset", bbox_to_anchor=(1.01, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(FIGS_DIR / "zero_shot_mae_bar.pdf", bbox_inches="tight")
    fig.savefig(FIGS_DIR / "zero_shot_mae_bar.png", bbox_inches="tight")
    plt.show()
    print("Saved zero_shot_mae_bar.pdf / .png")
else:
    print("No zero-shot data to plot.")


In [ ]:
# 9-B  Zero-shot vs Few-shot MAE comparison (MOMENT + Lag-Llama)
import matplotlib.pyplot as plt
import numpy as np

if not df_zs.empty and not df_fs.empty:
    fs_models = [m for m in MODELS_FEW_SHOT if m in df_fs["model"].values]
    fig, axes = plt.subplots(1, len(fs_models), figsize=(5*len(fs_models), 4),
                             sharey=False)
    if len(fs_models) == 1:
        axes = [axes]

    for ax, model in zip(axes, fs_models):
        zs_row = df_zs[df_zs["model"] == model].set_index("dataset")["mae"]
        fs_row = df_fs[df_fs["model"] == model].set_index("dataset")["mae"]

        common_ds = sorted(set(zs_row.index) & set(fs_row.index))
        if not common_ds:
            ax.set_title(f"{model}\n(no common datasets)")
            continue

        x = np.arange(len(common_ds))
        ax.bar(x - 0.2, [zs_row.get(d, np.nan) for d in common_ds],
               0.35, label="Zero-shot", color="steelblue", alpha=0.85)
        ax.bar(x + 0.2, [fs_row.get(d, np.nan) for d in common_ds],
               0.35, label="Few-shot", color="coral", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(common_ds, rotation=15, ha="right")
        ax.set_ylabel("MAE")
        ax.set_title(f"{model.upper()}: ZS vs FS (1%)")
        ax.legend()

    fig.tight_layout()
    fig.savefig(FIGS_DIR / "zs_vs_fs_comparison.pdf", bbox_inches="tight")
    fig.savefig(FIGS_DIR / "zs_vs_fs_comparison.png", bbox_inches="tight")
    plt.show()
    print("Saved zs_vs_fs_comparison.pdf / .png")
else:
    print("Need both zero-shot and few-shot results to plot comparison.")


In [ ]:
# 9-C  Cross-condition heatmap (MAE)
import matplotlib.pyplot as plt
import seaborn as sns

if not df_cc.empty:
    models_cc = [m for m in MODELS_ZERO_SHOT if m in df_cc["model"].values]
    targets   = sorted(df_cc["target"].unique())

    mat = []
    for model in models_cc:
        row = []
        for tgt in targets:
            val = df_cc[(df_cc["model"] == model) & (df_cc["target"] == tgt)]["mae"].mean()
            row.append(val)
        mat.append(row)

    import numpy as np
    mat = np.array(mat, dtype=float)

    fig, ax = plt.subplots(figsize=(max(4, len(targets) * 1.5), max(3, len(models_cc))))
    sns.heatmap(mat, annot=True, fmt=".4f",
                xticklabels=targets, yticklabels=models_cc,
                cmap="YlOrRd", ax=ax,
                cbar_kws={"label": "MAE"})
    ax.set_title("Cross-Condition MAE: C-MAPSS FD001 → FD00x")
    ax.set_xlabel("Target Condition")
    ax.set_ylabel("Model")
    fig.tight_layout()
    fig.savefig(FIGS_DIR / "cross_condition_heatmap.pdf", bbox_inches="tight")
    fig.savefig(FIGS_DIR / "cross_condition_heatmap.png", bbox_inches="tight")
    plt.show()
    print("Saved cross_condition_heatmap.pdf / .png")
else:
    print("No cross-condition data to plot.")


## 💾 Section 10 — Export to Google Drive

In [ ]:
# 10-A  Copy all results + figures + tables to Drive
import shutil
from pathlib import Path
from datetime import datetime

if DRIVE_AVAILABLE and DRIVE_RESULTS_DIR:
    drive_root = Path(DRIVE_RESULTS_DIR)
    drive_root.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    for src_rel in ["results/zero_shot", "results/few_shot",
                    "results/cross_condition", "results/tables", "results/figures"]:
        src = Path(src_rel)
        if src.exists():
            dst = drive_root / src.relative_to("results")
            shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
            print(f"  Copied {src_rel} → Drive")

    # Also save a timestamped summary CSV
    if not df_all.empty:
        summary_path = drive_root / f"summary_{timestamp}.csv"
        df_all.to_csv(summary_path, index=False)
        print(f"  Summary CSV → {summary_path}")
else:
    print("Drive not available — results remain in /content/tsfm-pdm-bench/results/")


In [ ]:
# 10-B  Final summary printout
print("=" * 70)
print("  EXPERIMENT SUMMARY")
print("=" * 70)

if not df_zs.empty:
    print(f"\nZero-shot runs:    {len(df_zs)}")
    print(f"  Avg MAE (all):    {df_zs['mae'].mean():.4f}")
if not df_fs.empty:
    print(f"\nFew-shot runs:     {len(df_fs)}")
    print(f"  Avg MAE (all):    {df_fs['mae'].mean():.4f}")
if not df_cc.empty:
    print(f"\nCross-cond runs:   {len(df_cc)}")
    print(f"  Avg MAE (FD001→): {df_cc['mae'].mean():.4f}")

print("\nOutput locations:")
print("  data/processed/          ← preprocessed .pt tensors")
print("  results/zero_shot/       ← per-run JSON files")
print("  results/few_shot/        ← per-run JSON files")
print("  results/cross_condition/ ← per-run JSON files")
print("  results/tables/          ← aggregated CSV summaries + efficiency profiles")
print("  results/figures/         ← PDF + PNG figures")
if DRIVE_AVAILABLE and DRIVE_RESULTS_DIR:
    print(f"  Google Drive:              {DRIVE_RESULTS_DIR}")
print("\nAll done — you have everything needed to write the article!")